# Customer Churn Prediction - Data Exploration

**Step 2: Perform initial data exploration and understanding**

This notebook covers:
- Loading the Telco Customer Churn dataset
- Examining data structure, types, and basic statistics
- Identifying target variable and feature columns
- Creating initial data quality report
- Documenting findings and insights

## Acceptance Criteria
- ✅ Dataset is successfully loaded and basic info is displayed
- ✅ Data quality issues are identified and documented
- ✅ Target variable distribution is analyzed
- ✅ Initial insights about the dataset are recorded
- ✅ Jupyter notebook with exploration is created

## 1. Setup and Imports

In [ ]:
# Standard library imports
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Data manipulation and analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Custom modules
sys.path.append('../src')
from data.data_loader import DataLoader
from utils.config import Config

# Configure plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

print("✅ All imports successful!")

## 2. Load Dataset

In [ ]:
# TODO: Initialize DataLoader and load the dataset
# Hint: Use DataLoader class from src.data.data_loader

# Initialize data loader
data_loader = DataLoader(data_path="../data")

# Load the Telco Customer Churn dataset
try:
    df = data_loader.load_telco_data()
    print(f"✅ Dataset loaded successfully!")
    print(f"📊 Dataset shape: {df.shape}")
except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print("💡 Please download the dataset first using scripts/download_data.py")
    # For demonstration, create sample data
    df = data_loader.load_sample_data()
    print(f"📊 Using sample data with shape: {df.shape}")

## 3. Initial Data Inspection

In [ ]:
# TODO: Display basic information about the dataset
# Examine first few rows, data types, and basic statistics

print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)

# Display first few rows
print("\n📋 First 5 rows:")
display(df.head())

# Display dataset info
print("\n📊 Dataset Info:")
df.info()

# Display basic statistics
print("\n📈 Basic Statistics:")
display(df.describe())

In [ ]:
# TODO: Examine data types and identify column categories
# Separate numerical and categorical columns

print("=" * 50)
print("COLUMN ANALYSIS")
print("=" * 50)

# Identify column types
numerical_columns = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = df.select_dtypes(include=['object']).columns.tolist()

print(f"\n🔢 Numerical columns ({len(numerical_columns)}):")
for col in numerical_columns:
    print(f"  - {col}")

print(f"\n📝 Categorical columns ({len(categorical_columns)}):")
for col in categorical_columns:
    unique_count = df[col].nunique()
    print(f"  - {col} ({unique_count} unique values)")

## 4. Data Quality Assessment

In [ ]:
# TODO: Check for missing values, duplicates, and data quality issues
# Create comprehensive data quality report

print("=" * 50)
print("DATA QUALITY ASSESSMENT")
print("=" * 50)

# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

print("\n❓ Missing Values:")
missing_df = pd.DataFrame({
    'Column': missing_values.index,
    'Missing Count': missing_values.values,
    'Missing Percentage': missing_percentage.values
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_df) > 0:
    display(missing_df)
else:
    print("✅ No missing values found!")

# Check for duplicates
duplicate_count = df.duplicated().sum()
print(f"\n🔄 Duplicate rows: {duplicate_count}")

# Check for unique identifier
if 'customerID' in df.columns:
    unique_customers = df['customerID'].nunique()
    total_rows = len(df)
    print(f"\n🆔 Unique customers: {unique_customers} out of {total_rows} rows")
    if unique_customers != total_rows:
        print("⚠️  Warning: Some customers may have multiple records")

## 5. Target Variable Analysis

In [ ]:
# TODO: Analyze the target variable (Churn) distribution
# Calculate churn rate and class balance

print("=" * 50)
print("TARGET VARIABLE ANALYSIS")
print("=" * 50)

if 'Churn' in df.columns:
    # Analyze churn distribution
    churn_counts = df['Churn'].value_counts()
    churn_percentages = df['Churn'].value_counts(normalize=True) * 100
    
    print("\n📊 Churn Distribution:")
    for value, count in churn_counts.items():
        percentage = churn_percentages[value]
        print(f"  {value}: {count} ({percentage:.1f}%)")
    
    # Calculate churn rate
    if 'Yes' in churn_counts.index:
        churn_rate = churn_percentages['Yes']
        print(f"\n📈 Overall Churn Rate: {churn_rate:.1f}%")
        
        # Assess class balance
        if churn_rate < 10 or churn_rate > 90:
            print("⚠️  Warning: Highly imbalanced dataset - consider sampling techniques")
        elif churn_rate < 20 or churn_rate > 80:
            print("⚠️  Moderately imbalanced dataset - monitor model performance")
        else:
            print("✅ Reasonably balanced dataset")
else:
    print("❌ 'Churn' column not found in dataset")

## 6. Feature Overview

In [ ]:
# TODO: Examine unique values in categorical columns
# Identify potential data quality issues

print("=" * 50)
print("CATEGORICAL FEATURES OVERVIEW")
print("=" * 50)

for col in categorical_columns:
    if col != 'customerID':  # Skip ID column
        unique_values = df[col].unique()
        print(f"\n📝 {col}:")
        print(f"  Unique values ({len(unique_values)}): {list(unique_values)}")
        
        # Check for potential issues
        if len(unique_values) > 20:
            print(f"  ⚠️  High cardinality - consider grouping or encoding strategy")
        
        # Check for inconsistent formatting
        if any(isinstance(val, str) and (val != val.strip() or val.lower() != val) for val in unique_values if pd.notna(val)):
            print(f"  ⚠️  Potential formatting issues detected")

In [ ]:
# TODO: Examine numerical features distribution
# Identify potential outliers and data ranges

print("=" * 50)
print("NUMERICAL FEATURES OVERVIEW")
print("=" * 50)

for col in numerical_columns:
    print(f"\n🔢 {col}:")
    print(f"  Range: {df[col].min():.2f} to {df[col].max():.2f}")
    print(f"  Mean: {df[col].mean():.2f}")
    print(f"  Median: {df[col].median():.2f}")
    print(f"  Std: {df[col].std():.2f}")
    
    # Check for potential outliers using IQR method
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    if len(outliers) > 0:
        outlier_percentage = (len(outliers) / len(df)) * 100
        print(f"  ⚠️  Potential outliers: {len(outliers)} ({outlier_percentage:.1f}%)")

## 7. Data Validation

In [ ]:
# TODO: Validate data schema and generate validation report
# Use DataLoader validation functionality

print("=" * 50)
print("DATA SCHEMA VALIDATION")
print("=" * 50)

# Validate data schema
validation_results = data_loader.validate_data_schema(df)

print(f"\n✅ Validation Status: {'PASSED' if validation_results['is_valid'] else 'FAILED'}")
print(f"📊 Dataset Shape: {validation_results['shape']}")
print(f"📋 Total Columns: {len(validation_results['columns'])}")

if validation_results['issues']:
    print("\n⚠️  Issues Found:")
    for issue in validation_results['issues']:
        print(f"  - {issue}")
else:
    print("\n✅ No validation issues found!")

## 8. Summary and Next Steps

In [ ]:
# TODO: Generate comprehensive data exploration summary
# Document key findings and recommendations

print("=" * 60)
print("DATA EXPLORATION SUMMARY")
print("=" * 60)

# Generate summary using DataLoader
data_info = data_loader.get_data_info(df)

print(f"\n📊 Dataset Overview:")
print(f"  - Shape: {data_info['shape']}")
print(f"  - Memory Usage: {data_info['memory_usage'] / 1024 / 1024:.2f} MB")
print(f"  - Missing Values: {data_info['missing_values']}")
print(f"  - Duplicate Rows: {data_info['duplicate_rows']}")

print(f"\n🎯 Key Findings:")
print(f"  - Total customers: {len(df)}")
print(f"  - Numerical features: {len(numerical_columns)}")
print(f"  - Categorical features: {len(categorical_columns)}")

if 'Churn' in df.columns:
    churn_rate = (df['Churn'] == 'Yes').mean() * 100
    print(f"  - Churn rate: {churn_rate:.1f}%")

print(f"\n📋 Next Steps:")
print(f"  1. Proceed to EDA notebook (02_eda_analysis.ipynb)")
print(f"  2. Create visualizations for deeper insights")
print(f"  3. Analyze relationships between features and churn")
print(f"  4. Identify patterns and business insights")

print(f"\n✅ Data exploration completed successfully!")